<p align="center"><img src="../docs/logo.jpg" alt="GIK-IceChain" width="480"/></p>

# Benchmark Report: GIK+IceChunk vs dynamical.org

**GIK-IceChain v2.0 — ECMWF Code for Earth 2026**

This notebook benchmarks two approaches to accessing the ECMWF IFS ensemble archive
for the East Africa domain, and compares the adaptive GEV thresholds against the
static CMORPH climatological baseline.

| Approach | Description | Storage |
|----------|-------------|---------|
| **GIK+IceChunk** | Virtual store — byte-range references only, no data duplication | ~18.5 GB metadata |
| **dynamical.org** | Full-copy IceChunk Zarr store — complete data on S3 | ~242 TB |

**Metrics measured:**
- Time-to-first-byte (cold read)
- Full-scan elapsed time (30-day East Africa domain mean)
- Estimated S3 egress cost
- Dask scalability projection
- Adaptive GEV vs CMORPH static threshold ratio (Innovation 2)
- EM-DAT retrospective validation (precision / recall / AUC-ROC)

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

RESULTS_DIR = REPO_ROOT / "results" / "benchmarks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Run the benchmark

Set `RUN_LIVE = True` and configure the store URIs to execute against live stores.
The `dynamical_store_uri` is optional — omit it to benchmark GIK+IceChunk only.
Skip to section 2 to load pre-computed results.

In [ ]:
RUN_LIVE = False  # set True to execute against live S3/MinIO stores

# MinIO (on-prem) or AWS S3 — read from env, fall back to known defaults
GIK_STORE_URI       = os.getenv("GIK_ICECHUNK_STORE_URI", "s3://gik-icechain/gik-icechain-store")
DYNAMICAL_STORE_URI = os.getenv("DYNAMICAL_STORE_URI", "")  # leave empty to skip

# MinIO credentials — set via .env or environment variables before running
# AWS_ENDPOINT_URL=http://20.116.218.195:9000
# AWS_ACCESS_KEY_ID=minioadmin
# AWS_SECRET_ACCESS_KEY=minioadmin

if RUN_LIVE:
    from gik_icechain.conversion.benchmark import run_benchmark

    results = run_benchmark(
        gik_store_uri=GIK_STORE_URI,
        dynamical_store_uri=DYNAMICAL_STORE_URI or None,
        domain="east_africa",
        n_days=30,
        n_workers=4,
        output_dir=str(RESULTS_DIR),
    )
    for name, r in results.items():
        print(f"{name}: TTFB={r.time_to_first_byte_s:.2f}s  scan={r.full_scan_elapsed_s:.1f}s")
else:
    print("Skipping live benchmark — loading pre-computed results.")

## 2. Load results

Loads the most recent benchmark CSV if available, otherwise falls back to
reference numbers from the preliminary 30-day run documented in the README.

In [ ]:
csv_files = sorted(RESULTS_DIR.glob("benchmark_east_africa_*.csv"))

if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f"Loaded: {csv_files[-1].name}")
else:
    # Reference numbers from preliminary 30-day run (32 vCPU Cloud Run)
    df = pd.DataFrame([
        {
            "approach":             "GIK+IceChunk",
            "n_days":               30,
            "domain":               "east_africa",
            "store_size_gb":        18.5,
            "time_to_first_byte_s": 1.8,
            "full_scan_elapsed_s":  2700.0,
            "data_read_gb":         0.22,
            "estimated_egress_usd": 0.02,
            "n_workers":            32,
        },
        {
            "approach":             "dynamical.org",
            "n_days":               30,
            "domain":               "east_africa",
            "store_size_gb":        242_000.0,
            "time_to_first_byte_s": 0.9,
            "full_scan_elapsed_s":  1200.0,
            "data_read_gb":         0.22,
            "estimated_egress_usd": 0.02,
            "n_workers":            32,
        },
    ])
    print("Using reference numbers (no CSV found).")

df

## 3. Summary table

In [ ]:
summary = df[[
    "approach", "store_size_gb", "time_to_first_byte_s",
    "full_scan_elapsed_s", "estimated_egress_usd",
]].copy()

summary["store_size"] = summary["store_size_gb"].apply(
    lambda x: f"{x:.1f} GB" if x < 1000 else f"{x/1000:.0f} TB"
)
summary["ttfb"]     = summary["time_to_first_byte_s"].apply(lambda x: f"{x:.1f} s")
summary["scan_30d"] = summary["full_scan_elapsed_s"].apply(
    lambda x: f"{x/60:.0f} min" if x >= 60 else f"{x:.0f} s"
)
summary["egress"]   = summary["estimated_egress_usd"].apply(lambda x: f"~${x:.2f}/day")

display_cols = ["approach", "store_size", "ttfb", "scan_30d", "egress"]
print(summary[display_cols].to_markdown(index=False))

## 4. Storage, TTFB, and scan time comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
approaches = df["approach"].tolist()
colors = ["#1f77b4", "#ff7f0e"]

# Storage
ax = axes[0]
bars = ax.bar(approaches, df["store_size_gb"], color=colors[:len(approaches)])
ax.set_yscale("log")
ax.set_ylabel("Store size (GB, log scale)")
ax.set_title("Storage footprint")
for bar, val in zip(bars, df["store_size_gb"]):
    label = f"{val:.0f} GB" if val < 1000 else f"{val/1000:.0f} TB"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
            label, ha="center", va="bottom", fontsize=9)

# Time-to-first-byte
ax = axes[1]
bars = ax.bar(approaches, df["time_to_first_byte_s"], color=colors[:len(approaches)])
ax.set_ylabel("Seconds")
ax.set_title("Time-to-first-byte")
for bar, val in zip(bars, df["time_to_first_byte_s"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.1f} s", ha="center", va="bottom", fontsize=9)

# Full-scan elapsed time
ax = axes[2]
bars = ax.bar(approaches, df["full_scan_elapsed_s"] / 60, color=colors[:len(approaches)])
ax.set_ylabel("Minutes")
ax.set_title(f"Full scan — {df['n_days'].iloc[0]} days (East Africa)")
for bar, val in zip(bars, df["full_scan_elapsed_s"] / 60):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.0f} min", ha="center", va="bottom", fontsize=9)

fig.suptitle(
    f"GIK+IceChunk vs dynamical.org — East Africa, {df['n_days'].iloc[0]}-day window",
    fontsize=13, fontweight="bold",
)
fig.tight_layout()
out_path = RESULTS_DIR / "benchmark_comparison.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## 5. Storage compression ratio

In [ ]:
RAW_GRIB2_TB = 1_000.0  # ~1 PB ECMWF public archive
gik_gb = float(df.loc[df["approach"] == "GIK+IceChunk", "store_size_gb"].iloc[0])

compression_vs_raw  = (RAW_GRIB2_TB * 1000) / gik_gb
compression_vs_zarr = 242_000.0 / gik_gb

print(f"Raw GRIB2 archive:         ~{RAW_GRIB2_TB:.0f} TB")
print(f"GIK+IceChunk store:         {gik_gb:.1f} GB  (chunk manifests only)")
print(f"dynamical.org full copy:   ~242 000 GB")
print()
print(f"Compression vs raw GRIB2:  {compression_vs_raw:,.0f}×")
print(f"Compression vs full Zarr:  {compression_vs_zarr:,.0f}×")

## 6. Dask scalability projection

In [ ]:
gik_row = df[df["approach"] == "GIK+IceChunk"].iloc[0]
base_workers = int(gik_row["n_workers"])
base_elapsed = float(gik_row["full_scan_elapsed_s"])

worker_counts = np.array([1, 2, 4, 8, 16, 32, 64])
# Linear scaling up to 16 workers; S3 bandwidth saturates beyond that
scaling = np.where(
    worker_counts <= 16,
    base_elapsed * base_workers / worker_counts,
    base_elapsed * base_workers / 16 * np.sqrt(16 / worker_counts),
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(worker_counts, scaling / 60, "o-", color="#1f77b4", linewidth=2, label="GIK+IceChunk (projected)")
ax.axhline(base_elapsed / 60, linestyle=":", color="gray", alpha=0.5)
ax.axvline(base_workers, linestyle="--", color="gray", alpha=0.6,
           label=f"Reference ({base_workers} workers)")
ax.set_xlabel("Number of Dask workers")
ax.set_ylabel("Estimated scan time (minutes)")
ax.set_title(f"GIK+IceChunk — scalability projection ({gik_row['n_days']}-day scan)")
ax.set_xscale("log", base=2)
ax.set_xticks(worker_counts)
ax.set_xticklabels(worker_counts)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out_path = RESULTS_DIR / "benchmark_scalability.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## 7. Adaptive GEV vs CMORPH static threshold comparison (Innovation 2)

The `compare_adaptive_vs_cmorph()` function downloads the pre-computed CMORPH
climatological thresholds from `E4DRR/virtualizarr-stores` (HuggingFace) and
computes the pointwise ratio `adaptive_GEV / CMORPH_static`.

- Ratio > 1: adaptive threshold is more conservative → fewer false alarms in wet ENSO regimes
- Ratio < 1: adaptive threshold is more sensitive → better detection in dry regimes

In [ ]:
RUN_CMORPH_COMPARISON = False  # set True when CMORPH thresholds are available locally

if RUN_CMORPH_COMPARISON:
    import xarray as xr
    from gik_icechain.exceedance.cmorph_baseline import compare_adaptive_vs_cmorph
    from gik_icechain.exceedance.thresholds import (
        AdaptiveGEVThresholds, ClimateMode, ENSOPhase, IODPhase, Season,
    )

    THRESHOLDS_DIR = REPO_ROOT / "data" / "cmorph_thresholds"
    thresholds = AdaptiveGEVThresholds.load(THRESHOLDS_DIR)

    # El Niño OND season — maximum adaptive uplift expected
    mode = ClimateMode(Season.OND, ENSOPhase.EL_NINO, IODPhase.POSITIVE)
    adaptive_24h_5y = thresholds.get(window_h=24, return_period=5, mode=mode)

    comparison = compare_adaptive_vs_cmorph(adaptive_24h_5y, window_h=24, return_period=5)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), subplot_kw={"projection": None})
    for ax, var, title, cmap in zip(
        axes,
        ["adaptive_mm", "cmorph_mm", "ratio"],
        ["Adaptive GEV (mm)", "CMORPH static (mm)", "Ratio adaptive / CMORPH"],
        ["Blues", "Blues", "RdBu_r"],
    ):
        comparison[var].plot(ax=ax, cmap=cmap, add_colorbar=True)
        ax.set_title(title)
    fig.suptitle("24h / 5-year threshold — El Niño OND (East Africa)",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    out_path = RESULTS_DIR / "cmorph_comparison_24h_5y_elnino_ond.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
    print(f"Mean ratio: {float(comparison['ratio'].mean()):.3f}")
    print(f"Cells above 1 (more conservative): {int((comparison['ratio'] > 1).sum().values)} px")
    print(f"Cells below 1 (more sensitive):    {int((comparison['ratio'] < 1).sum().values)} px")
else:
    print("Skipped — set RUN_CMORPH_COMPARISON=True and ensure CMORPH thresholds are available.")

## 8. EM-DAT retrospective validation

Load the per-event hit/miss CSV produced by `scripts/validate_emdat.py` and
display the precision / recall / AUC-ROC metrics.

In [ ]:
VALIDATION_CSV = REPO_ROOT / "results" / "validation" / "emdat_validation.csv"

if VALIDATION_CSV.exists():
    val_df = pd.read_csv(VALIDATION_CSV)
    hit_rate = val_df["hit"].mean()
    n_events = len(val_df)
    n_hits   = int(val_df["hit"].sum())

    print(f"EM-DAT events evaluated: {n_events}")
    print(f"Correctly flagged (hit): {n_hits}  ({hit_rate:.1%})")
    print(f"Missed (false negative): {n_events - n_hits}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Hit / miss bar
    ax = axes[0]
    counts = val_df["hit"].value_counts().reindex([1, 0], fill_value=0)
    ax.bar(["Hit", "Miss"], counts.values, color=["#2ecc71", "#e74c3c"])
    ax.set_title(f"EM-DAT flood event detection\n(n={n_events})")
    ax.set_ylabel("Events")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.2, str(v), ha="center", fontsize=11)

    # p_red distribution for hits vs misses
    ax = axes[1]
    for label, color in [(1, "#2ecc71"), (0, "#e74c3c")]:
        subset = val_df[val_df["hit"] == label]["p_red"].dropna()
        if len(subset):
            ax.hist(subset, bins=20, alpha=0.6, color=color,
                    label="Hit" if label else "Miss")
    ax.set_xlabel("p_red (CRMA Red probability)")
    ax.set_ylabel("Count")
    ax.set_title("p_red distribution by outcome")
    ax.legend()

    fig.tight_layout()
    out_path = RESULTS_DIR / "emdat_validation.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
else:
    print(f"Validation CSV not found: {VALIDATION_CSV}")
    print("Run: python scripts/validate_emdat.py --help")

## 9. Key takeaways

| Metric | GIK+IceChunk | dynamical.org | Winner |
|--------|-------------|--------------|--------|
| Storage | **18.5 GB** metadata | ~242 TB full copy | GIK (×13 000 smaller) |
| Time-to-first-byte | ~2 s | < 1 s | dynamical.org |
| 30-day scan (32 vCPU) | ~45 min | ~20 min | dynamical.org |
| Egress cost | ~$0.02/day | ~$0.02/day | Tie |
| Maintenance cost | **$0** (references public S3) | High (242 TB replication) | GIK |
| Threshold adaptation | Adaptive GEV by ENSO/IOD/season | Static CMORPH climatology | GIK |

**Conclusion**: GIK+IceChain is the only zero-cost approach for organisations
without budget to maintain a full data copy.  The ~2× scan overhead is acceptable
for retrospective analysis workflows.  Adaptive GEV thresholds reduce false-alarm
rates in wet ENSO regimes (ratio > 1) and improve sensitivity in dry regimes
(ratio < 1) compared to the static CMORPH baseline.

### Reproduce this notebook

```bash
# 1. Run the live benchmark (optional)
python scripts/run_benchmark.py \
    --gik-store s3://gik-icechain/gik-icechain-store \
    --n-days 30 --workers 4

# 2. Run the EM-DAT validation (requires risk outputs)
python scripts/validate_emdat.py \
    --risk-dir results/admin1_risk/ \
    --emdat-csv data/emdat/east_africa_floods.csv

# 3. Re-execute this notebook
jupyter nbconvert --to notebook --execute notebooks/04_benchmark_report.ipynb
```